In [1]:
import tensorflow as tf
print(tf.__version__)


2026-01-07 02:52:03.506687: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767754323.519666      11 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767754323.523648      11 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-01-07 02:52:03.539870: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

2.18.0


In [2]:
import numpy as np
import pandas as pd
import json

def read_lines_m(path, max_limit=4000):
    rlm = []; ml = max_limit
    for l in open(path, 'r'):
        rlm.append(json.loads(l))
        ml -= 1
        if ml <= 0: break
    return pd.DataFrame(rlm)

p = '../input/tensorflow2-question-answering/'
train = read_lines_m(p + 'simplified-nq-train.jsonl')
train['D'] = [t[0]['long_answer']['start_token'] for t in train.annotations]
train = train[train['D']>-1].reset_index(drop=True)
test = read_lines_m(p + 'simplified-nq-test.jsonl').reset_index(drop=True)
sub = pd.read_csv(p + 'sample_submission.csv')
train.shape, test.shape, sub.shape


((2068, 7), (4000, 4), (61476, 2))

In [3]:
i =99
print('URL:', train.document_url[i])
print(train.question_text[i])
print(train.long_answer_candidates[i][0])
print(' '.join(train.document_text[i].split()[train.annotations[i][0]['long_answer']['start_token'] : train.annotations[i][0]['long_answer']['end_token']]))
if len(train.annotations[i][0]['short_answers']) > 0:
    print(' '.join(train.document_text[i].split()[train.annotations[i][0]['short_answers'][0]['start_token'] : train.annotations[i][0]['short_answers'][0]['end_token']]))


URL: https://en.wikipedia.org//w/index.php?title=Petty_officer&amp;oldid=807097820
where does the term petty officer come from
{'start_token': 14, 'top_level': True, 'end_token': 65}
<P> The modern petty officer dates back to the Age of Sail . Petty officers rank between naval officers ( both commissioned and warrant ) and most enlisted sailors . These were men with some claim to officer rank , sufficient to distinguish them from ordinary ratings , without raising them so high as the sea officers . Several were warrant officers , in the literal sense of being appointed by warrant , and like the warrant sea officers , their superiors , they were usually among the specialists of the ship 's company . The Oxford English Dictionary suggests that the title derives from the Anglo - Norman and Middle French `` petit '' , meaning `` of small size , small , little '' . </P>
Age of Sail


In [4]:
la=[t[0]['long_answer']['end_token'] - t[0]['long_answer']['start_token'] for t in train.annotations]
sa=[t[0]['short_answers'][0]['end_token'] - t[0]['short_answers'][0]['start_token'] for t in train.annotations if len(t[0]['short_answers'])>0]
np.median(la), np.median(sa)


(118.0, 2.0)

In [5]:
from bs4 import BeautifulSoup as b
import random

result = []
for i in range(len(test.example_id)):
    s = b(test.document_text[i], 'html.parser')
    p = [p.get_text() for p in s.find_all('p', text=True) if len(p) > 50]
    if len(p)>0:
        r = test.document_text[i].find(p[0]) #optimize for better results
        r = len(test.document_text[i][:r].split()) - 1
        long_answer= ''.join([str(r),':', str(r + len(p[0].split()) + 2)])
    else:
        try:
            r = random.randrange(390, len(test.document_text[i].split()))
        except:
            r=7
        long_answer= ''.join([str(r),':', str(r + 114)])

    if len([q for q in ['am', 'are', 'can', 'could', 'did', 'do', 'does', 'has', 'have', 'is', 'may', 'should', 'was', 'were', 'will'] if q in test.question_text[i].lower().split()])>0:
        short_answer=random.choice(['YES','NO'])
    else:
        r = random.randrange(r, r + 114)
        short_answer=''.join([str(r),':', str(r + 2)])
    result.append([test.example_id[i] + '_long', long_answer])
    result.append([test.example_id[i] + '_short', short_answer])
pd.DataFrame(result,columns=['example_id', 'PredictionString']).to_csv('submission.csv', index=False)


/tmp/ipykernel_11/927017910.py:7: DeprecationWarning: The 'text' argument to find()-type methods is deprecated. Use 'string' instead.
  p = [p.get_text() for p in s.find_all('p', text=True) if len(p) > 50]


In [6]:
pd.DataFrame(result,columns=['example_id', 'PredictionString']).head()


,example_id,PredictionString
0,-257485578111885185_long,1412:1526
1,-257485578111885185_short,1465:1467
2,-9110190923673509457_long,758:872
3,-9110190923673509457_short,NO
4,-2606047177489670354_long,3356:3470
